# Text Classification Datasets for LLM Calibration - Demo Notebook

## Overview

This notebook demonstrates how to load and transform text classification datasets for LLM calibration evaluation. 

### What this notebook does:
- Loads curated text classification datasets from HuggingFace Hub
- Standardizes dataset format to a unified JSON schema
- Transforms raw datasets into the `exp_sel_data_out.json` format
- Displays dataset statistics and sample examples

### Datasets included:
- **SST-2**: Binary sentiment classification (positive/negative)
- **AG News**: 4-class topic classification (World, Sports, Business, Sci/Tech)
- **MNLI**: 3-class natural language inference (entailment, neutral, contradiction)
- **QNLI**: 2-class question-answer entailment
- **DBpedia**: 14-class ontology classification

### Original artifact:
- Generated 150,000 total examples across 5 datasets
- Full, mini (15 examples), and preview variants available
- All files passed JSON schema validation

## Cell 1: Install Dependencies

Install required packages. On Colab, core packages (numpy, pandas, sklearn, etc.) are pre-installed so we skip them. Locally, we install at Colab's exact versions to match the environment.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# datasets library - NOT pre-installed on Colab, always install
_pip('datasets==4.0.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

## Cell 2: Imports

Import all required libraries. The original `data.py` script uses `json` and `pathlib.Path`.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

print("Imports complete.")

## Cell 3: Data Loading Helper

Load the demo data from GitHub (with local fallback). The `mini_demo_data.json` contains a curated subset of the full dataset.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d144fc-thermodynamic-entropy-calibration-for-im/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub download failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os
data = load_data()
print(f"Loaded data with {data['metadata']['num_datasets']} datasets")
print(f"Total examples: {data['metadata']['total_examples']}")

## Cell 4: Configuration

Define tunable parameters. For this demo, we use minimal values to keep runtime fast.

**Original parameters:**
- Full dataset: 150,000 examples across 5 datasets
- Demo uses: 10 examples from 1 dataset (SST-2)

**To scale up:** Change `DEMO_MODE` to `False` and adjust `MAX_EXAMPLES_PER_DATASET`.

In [ ]:
# Configuration - MINIMAL values for fast demo
DEMO_MODE = True  # Set to False to use more data
MAX_EXAMPLES_PER_DATASET = 10 if DEMO_MODE else 100  # Max examples to load per dataset
DATASETS_TO_USE = ['sst-2']  # For demo, use only SST-2; original uses all 5 datasets

print(f"Demo mode: {DEMO_MODE}")
print(f"Max examples per dataset: {MAX_EXAMPLES_PER_DATASET}")
print(f"Datasets to use: {DATASETS_TO_USE}")

## Cell 5: Transform Dataset Function

This is the core transformation function from `data.py`. It converts the raw dataset format to the unified schema with:
- `input`: The text input
- `output`: String label (0-indexed)
- `metadata_label_text`: Human-readable label
- `metadata_original_label`: Original integer label

In [ ]:
def transform_dataset(input_file, dataset_name):
    """
    Transform dataset to unified format.
    
    Original function from data.py:
    - Loads JSON from input_file
    - Extracts text, label, label_text, and metadata
    - Returns standardized format
    
    For demo: we use the already-loaded `data` variable instead of reading from file.
    """
    # In demo mode, we use the pre-loaded data variable
    if DEMO_MODE:
        # Find the dataset in pre-loaded data
        for dataset in data['datasets']:
            if dataset['dataset'] == dataset_name:
                examples = dataset['examples'][:MAX_EXAMPLES_PER_DATASET]
                return {"dataset": dataset_name, "examples": examples}
        return None
    else:
        # Original behavior: load from file
        with open(Path(input_file), "r") as f:
            data = json.load(f)
        examples = []
        for row in data:
            example = {
                "input": row["text"],
                "output": str(row["label"]),
                "metadata_label_text": row.get("label_text", ""),
                "metadata_original_label": row["metadata"]["original_label"],
            }
            examples.append(example)
        return {"dataset": dataset_name, "examples": examples}

print("Transform function defined.")

## Cell 6: Process All Datasets

Run the transformation on all configured datasets. In demo mode, this processes only the SST-2 dataset with 10 examples.

In [ ]:
# Process datasets
all_datasets = []

for dataset_name in DATASETS_TO_USE:
    print(f"\nTransforming {dataset_name}...")
    
    # In demo mode, we don't need input_file (uses pre-loaded data)
    dataset_group = transform_dataset(None, dataset_name)
    
    if dataset_group:
        all_datasets.append(dataset_group)
        print(f"  Added {len(dataset_group['examples'])} examples")
    else:
        print(f"  Warning: Dataset {dataset_name} not found in data")

# Create output structure
output = {
    "datasets": all_datasets,
    "metadata": {
        "description": "Text classification datasets for LLM calibration",
        "num_datasets": len(all_datasets),
        "total_examples": sum(len(d["examples"]) for d in all_datasets)
    }
}

print(f"\nProcessed {output['metadata']['num_datasets']} datasets")
print(f"Total examples: {output['metadata']['total_examples']}")

## Cell 7: Visualize Results

Display dataset statistics, sample examples, and label distributions.

In [ ]:
# Display results
print("="*60)
print("DATASET SUMMARY")
print("="*60)

for dataset in output['datasets']:
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    print(f"\nDataset: {dataset_name}")
    print(f"  Number of examples: {len(examples)}")
    
    # Count labels
    label_counts = Counter(ex['output'] for ex in examples)
    print(f"  Label distribution: {dict(label_counts)}")
    
    # Show sample examples
    print(f"  Sample examples:")
    for i, ex in enumerate(examples[:3]):
        print(f"    {i+1}. Input: \"{ex['input'][:50]}...\"")
        print(f"       Output: {ex['output']} ({ex['metadata_label_text']})")

# Visualize label distribution
print("\n" + "="*60)
print("LABEL DISTRIBUTION VISUALIZATION")
print("="*60)

fig, axes = plt.subplots(1, len(output['datasets']), figsize=(5*len(output['datasets']), 4))
if len(output['datasets']) == 1:
    axes = [axes]

for idx, dataset in enumerate(output['datasets']):
    examples = dataset['examples']
    label_counts = Counter(ex['metadata_label_text'] for ex in examples)
    
    # Sort by label name for consistent ordering
    labels = sorted(label_counts.keys())
    counts = [label_counts[label] for label in labels]
    
    ax = axes[idx]
    bars = ax.bar(labels, counts, color='skyblue', edgecolor='black')
    ax.set_title(f"{dataset['dataset']}\n({len(examples)} examples)")
    ax.set_xlabel("Label")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=45)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Show metadata
print("\n" + "="*60)
print("METADATA")
print("="*60)
print(json.dumps(output['metadata'], indent=2))

## Cell 8: Save Output (Optional)

Save the transformed data to a JSON file. This mirrors the original `data.py` behavior which saves to `full_data_out.json`.

In [ ]:
# Save output to JSON file (optional)
output_file = "demo_output.json"

with open(output_file, "w") as f:
    json.dump(output, f, indent=2)

print(f"Output saved to: {output_file}")
print(f"File size: {Path(output_file).stat().st_size} bytes")

# Display first few lines of output file
print("\nFirst 500 characters of output file:")
with open(output_file) as f:
    print(f.read()[:500] + "...")